# Event-driven backtests: bars, tape, and quotes

`BacktestSignal` (notebook 14) marks a position series to a single price. The
rest of the backtest family models how the order actually fills against a
richer view of the market: OHLC bars, the trade tape, or top-of-book quotes.
All four share the `[equity, pnl, position, cost]` output and `backtest_report`,
and all are causal, so the same code runs live.

Here we drive the three market-data engines from one real Deribit
BTC-perpetual trade tape.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer import (BacktestOHLC, BacktestTrades, BacktestL1, BacktestL1Trades,
                      RollingMean, Resample, Lag, backtest_report)

trades = pd.read_csv("data/deribit_btc_perp_6h.csv")
ts = trades["timestamp"].to_numpy(np.int64)
price = trades["price"].to_numpy(np.float64)
size = np.abs(trades["volume"].to_numpy(np.float64))   # signed in the file; magnitude here
print(f"{len(price)} prints over {(ts[-1]-ts[0])/3.6e6:.1f} hours")

## Bars: `BacktestOHLC`

Resample the tape into one-minute OHLC bars and trade a moving-average-crossover
target with market orders (a `NaN` limit price fills at the open). `BacktestOHLC`
is causal by design: the target computed from a bar's close is deferred and traded
on the next bar, so we feed the raw signal with no manual lag.

In [ ]:
o, idx = Resample(freq=60_000, agg="first")(price, ts)   # one-minute OHLC
h, _   = Resample(freq=60_000, agg="max")(price, ts)
l, _   = Resample(freq=60_000, agg="min")(price, ts)
c, _   = Resample(freq=60_000, agg="last")(price, ts)
tb = pd.to_datetime(idx, unit="ms")

fast, slow = RollingMean(5)(c), RollingMean(30)(c)
signal = np.sign(np.nan_to_num(fast - slow))            # decided on each close; engine defers it
market = np.full(len(c), np.nan)                        # market-on-open next bar

ohlc = BacktestOHLC(spread=0.0005, taker_fee=0.0002)(signal, market, o, h, l, c)
print(backtest_report(ohlc)[1].round(4).to_string())

## Tape: `BacktestTrades`

Now work event by event on the raw prints. A mean-reverting maker rests a buy
one tick below the print when price is under a rolling mid and a sell one tick
above it when price is over the mid; each resting order is lagged one event, so
it fills only when a *later* print trades through it. Fills accumulate inventory
that this engine leaves uncapped, quoting the tape is a directional bet unless
you bound it yourself (which is exactly what `BacktestL1` adds below).

In [ ]:
mid = np.nan_to_num(RollingMean(50)(price), nan=price[0])   # a smooth reference
buy = (price - mid) < 0                                     # buy under the mid, sell over it
order_price = np.where(buy, price - 1.0, price + 1.0)       # rest one tick inside
order_size = np.where(buy, 1.0, -1.0)                       # signed: + buy, - sell
# the order rests from the previous event; lag it against the current print
op = np.nan_to_num(Lag(1)(order_price), nan=0.0)
os_ = np.nan_to_num(Lag(1)(order_size), nan=0.0)

tp = BacktestTrades(maker_fee=-0.0001)(op, os_, price, size)
eq, pos = tp[:, 0], tp[:, 2]
tt = pd.to_datetime(ts, unit="ms")

fig, (a0, a1) = plt.subplots(2, 1, sharex=True, figsize=(9, 5),
                             gridspec_kw={"height_ratios": [2, 1]})
a0.plot(tt, eq, color="steelblue"); a0.axhline(0, color="k", lw=0.5)
a0.set_ylabel("equity ($)"); a0.set_title("BacktestTrades: a mean-reverting maker on the tape")
a1.plot(tt, pos, color="darkorange"); a1.set_ylabel("inventory (uncapped)")
fig.tight_layout()

## Quotes only: `BacktestL1`

With quotes but no trades, fills are a documented heuristic. The default `breach`
fills only when the market trades through your quote; `touch` also captures a
participation share once per lock. We rest last event's touch on both sides and
bound inventory to +/-15. Fills here can over- or under-count because a quote-size
change cannot be told from a cancel, which is exactly what the trade feed fixes
below.

In [ ]:
half = 1.0
bid, ask = price - half, price + half
# rest last event's touch; a market move through it is the fill (Lag avoids lookahead)
my_bid = np.nan_to_num(Lag(1)(bid), nan=bid[0])
my_ask = np.nan_to_num(Lag(1)(ask), nan=ask[0])
one, five = np.ones(len(price)), np.full(len(price), 5.0)

l1 = BacktestL1(fill="touch", maker_fee=-0.0001, participation_ratio=0.5,
                max_position=15.0, min_position=-15.0)(
    bid, ask, five, five, my_bid, one, my_ask, one)
eq1, pos1 = l1[:, 0], l1[:, 2]

fig, (a0, a1) = plt.subplots(2, 1, sharex=True, figsize=(9, 5),
                             gridspec_kw={"height_ratios": [2, 1]})
a0.plot(tt, eq1, color="steelblue"); a0.axhline(0, color="k", lw=0.5)
a0.set_ylabel("equity ($)"); a0.set_title("BacktestL1: quotes-only maker (heuristic fills)")
a1.plot(tt, pos1, color="mediumpurple"); a1.axhline(15, color="0.7", lw=0.5, ls="--")
a1.axhline(-15, color="0.7", lw=0.5, ls="--"); a1.set_ylabel("inventory")
fig.tight_layout()

## Quotes + trades: `BacktestL1Trades`

The preferred engine. Quotes mark the position and the real trade tape drives the
fills, so there is no fill-versus-cancel ambiguity. We feed the same lagged quotes
plus the actual prints; each trade fills at most once. This is the honest
market-making backtest of the three.

In [ ]:
l1t = BacktestL1Trades(fill="touch", maker_fee=-0.0001, participation_ratio=0.3,
                       max_position=15.0, min_position=-15.0)(
    bid, ask, five, five, my_bid, one, my_ask, one, price, size)   # price/size = the real tape
eqt, post = l1t[:, 0], l1t[:, 2]

fig, (a0, a1) = plt.subplots(2, 1, sharex=True, figsize=(9, 5),
                             gridspec_kw={"height_ratios": [2, 1]})
a0.plot(tt, eqt, color="steelblue"); a0.axhline(0, color="k", lw=0.5)
a0.set_ylabel("equity ($)"); a0.set_title("BacktestL1Trades: trade-driven maker")
a1.plot(tt, post, color="seagreen"); a1.set_ylabel("inventory")
fig.tight_layout()

## The same statistics everywhere

Every engine emits `[equity, pnl, position, cost]`, so `backtest_report` reads
the same summary off any of them. Pick the engine that matches the market data
you have; the accounting, costs, and reporting are shared.

In [ ]:
for name, out in [("OHLC", ohlc), ("Trades", tp), ("L1", l1), ("L1Trades", l1t)]:
    s = backtest_report(out)[1]
    print(f"{name:9s}  pnl={s['total_pnl']:10.2f}  cost={s['total_cost']:9.2f}  "
          f"trades={s['num_trades']:6.0f}  maxDD={s['max_drawdown']:9.2f}")